# 1. Log & Trace Analysis

Analyze SLURM logs and trace.json files from Harmonia experiments using the CLI tool.

**Failure mode taxonomy:** 13 categories defined in `types_of_log_and_trace_problems.yaml`

In [ ]:
import os, subprocess, json, textwrap
os.chdir("/hpc/compgen/projects/llm_GEO_project/harmonia_metadata_agent/analysis/dstoker/harmonia")

CLI = "code_development_tools_agents/monitoring_and_evaluation/read_and_analyze_logs_and_traces_cli.py"
PYTHON = ".venv/bin/python"

## Quick summary of recent runs

In [ ]:
result = subprocess.run([PYTHON, CLI], capture_output=True, text=True)
print(result.stdout)
if result.stderr:
    print("STDERR (last 2000 chars):", result.stderr[-2000:])

## Verbose per-turn trace analysis

In [ ]:
result = subprocess.run([PYTHON, CLI, "--verbose"], capture_output=True, text=True)
print(result.stdout)
if result.stderr:
    print("STDERR (last 2000 chars):", result.stderr[-2000:])

## JSON output for programmatic analysis

In [ ]:
result = subprocess.run([PYTHON, CLI, "--json"], capture_output=True, text=True)
try:
    data = json.loads(result.stdout)
    for run in data.get("runs", []):
        run_id = run.get("run_id", "?")
        exp = run.get("experiment_name", "?")
        problems = run.get("problems", [])
        status = "OK" if not problems else f"{len(problems)} issue(s)"
        severities = [p.get("severity", "?") for p in problems]
        print(f"  {run_id} | {exp:50s} | {status:15s} | {severities}")
except json.JSONDecodeError:
    print("No valid JSON output")
    print(result.stdout[:2000])
    if result.stderr:
        print("STDERR:", result.stderr[-1000:])

## Analyze a specific run by ID

Change `RUN_ID` below to inspect a particular experiment.

In [ ]:
RUN_ID = ""  # e.g. "a1b2c3d4"

if RUN_ID:
    result = subprocess.run([PYTHON, CLI, "--run-id", RUN_ID, "--verbose"],
                            capture_output=True, text=True)
    print(result.stdout)
    if result.stderr:
        print("STDERR:", result.stderr[-2000:])
else:
    print("Set RUN_ID above to analyze a specific run.")

## With diagnostics (RCA)

In [ ]:
result = subprocess.run([PYTHON, CLI, "--diagnostics"], capture_output=True, text=True)
print(result.stdout)
if result.stderr:
    print("STDERR (last 2000 chars):", result.stderr[-2000:])